In [2]:
import pandas as pd
import glob
import os
import numpy as np

ruta = "D:/Licit/1/ML-SmartBids/csv/*.csv"

archivos = sorted(glob.glob(ruta))

print(f"Archivos encontrados: {len(archivos)}")

if len(archivos) == 0:
    raise FileNotFoundError(
        "No se encontraron archivos CSV."
    )

dataframes = []

for archivo in archivos:

    nombre = os.path.basename(archivo)

    print("\n" + "=" * 80)
    print(f"Leyendo: {nombre}")
    print("=" * 80)

    df_mes = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        encoding_errors="replace",
        low_memory=False
    )

    print(f"Filas originales: {len(df_mes):,}")

    # --------------------------------------------------------
    # Detectar problemas de codificación
    # --------------------------------------------------------

    columnas_texto = df_mes.select_dtypes(
        include=["object", "string"]
    ).columns

    for columna in columnas_texto:

        mascara = (
            df_mes[columna]
            .astype("string")
            .str.contains("�", regex=False, na=False)
        )

        cantidad = mascara.sum()

        if cantidad > 0:

            print(
                f"  ⚠ {columna}: "
                f"{cantidad:,} valores corruptos -> NaN"
            )

            df_mes.loc[mascara, columna] = np.nan

    # --------------------------------------------------------
    # Revisar NaN de oferta
    # --------------------------------------------------------

    mask_sin_oferta = (
        df_mes["Cantidad Ofertada"].isna()
        & df_mes["Valor Total Ofertado"].isna()
    )

    print(
        f"Filas sin oferta: "
        f"{mask_sin_oferta.sum():,}"
    )

    # --------------------------------------------------------
    # Eliminar filas sin oferta
    # --------------------------------------------------------

    df_mes = df_mes.loc[~mask_sin_oferta].copy()

    print(
        f"Filas conservadas: "
        f"{len(df_mes):,}"
    )

    dataframes.append(df_mes)


# ============================================================
# UNIR
# ============================================================

df = pd.concat(
    dataframes,
    ignore_index=True
)

# Liberar referencias a los DataFrames mensuales
del dataframes


# ============================================================
# COMPROBACIONES
# ============================================================

print("\n" + "=" * 80)
print("DATASET ANUAL")
print("=" * 80)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns):,}")
print(f"Dimensiones: {df.shape}")

print("\nNaN en variables de oferta:")

print(
    df[
        [
            "Cantidad Ofertada",
            "Valor Total Ofertado"
        ]
    ].isna().sum()
)


# ============================================================
# GUARDAR
# ============================================================

df.to_feather(
    "licitaciones_anual.feather"
)

print("\n✓ Guardado como licitaciones_anual.feather")

Archivos encontrados: 12

Leyendo: lic_2024-01.csv
Filas originales: 203,486
  ⚠ Descripcion linea Adquisicion: 9 valores corruptos -> NaN
  ⚠ DescripcionProveedor: 11 valores corruptos -> NaN
Filas sin oferta: 0
Filas conservadas: 203,486

Leyendo: lic_2024-02.csv
Filas originales: 216,727
  ⚠ Descripcion linea Adquisicion: 4 valores corruptos -> NaN
  ⚠ DescripcionProveedor: 4 valores corruptos -> NaN
Filas sin oferta: 8
Filas conservadas: 216,719

Leyendo: lic_2024-03.csv
Filas originales: 220,947
  ⚠ DescripcionProveedor: 9 valores corruptos -> NaN
Filas sin oferta: 5
Filas conservadas: 220,942

Leyendo: lic_2024-04.csv
Filas originales: 254,411
  ⚠ DescripcionProveedor: 4 valores corruptos -> NaN
Filas sin oferta: 38
Filas conservadas: 254,373

Leyendo: lic_2024-05.csv
Filas originales: 200,639
  ⚠ Descripcion linea Adquisicion: 1 valores corruptos -> NaN
  ⚠ DescripcionProveedor: 5 valores corruptos -> NaN
Filas sin oferta: 110
Filas conservadas: 200,529

Leyendo: lic_2024-06.csv

In [ ]:
for archivo in archivos:

    nombre = os.path.basename(archivo)

    df_test = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        encoding_errors="replace",
        low_memory=False
    )

    print("\n", "=" * 80)
    print(nombre)

    for col in df_test.columns:
        if "ofert" in str(col).lower():
            print(repr(col))